In [1]:
import os
import json
import pandas as pd
from typing import Dict, List, Any, Optional
from IPython.display import display

In [2]:
# Configuration and constants
TARGET_PAIRS: List[str] = ['en-ko', 'en-zh']
MAIN_FILE: str = '../data/wmt25/wmt25-genmt.jsonl'
SYS_DIR: str = '../data/wmt25/systems/'
BASE_OUT_DIR: str = '../data/wmt25/'

In [3]:
def parse_reference_text(refs_data: Any) -> str:
    """Extract plain text from complex nested reference dictionaries."""
    if isinstance(refs_data, dict) and refs_data:
        ref_val = list(refs_data.values())[0] 
        if isinstance(ref_val, dict):
            return str(list(ref_val.values())[0])
        elif isinstance(ref_val, list):
            return '\n'.join(map(str, ref_val))
        return str(ref_val)
    return str(refs_data)

In [4]:
def normalize_lang_pair(doc_id: str) -> Optional[str]:
    """Extract and normalize target language pair from document ID."""
    if doc_id.startswith('en-ko'): return 'en-ko'
    if doc_id.startswith('en-zh'): return 'en-zh'
    return None

In [5]:
# Load system hypotheses into memory
system_files: List[str] = [f for f in os.listdir(SYS_DIR) if f.endswith('.jsonl')]
sys_answers: Dict[str, Dict[str, str]] = {}

In [6]:
for sys_file in system_files:
    sys_name: str = sys_file.replace('.jsonl', '')
    sys_answers[sys_name] = {}
    with open(os.path.join(SYS_DIR, sys_file), 'r', encoding='utf-8') as f:
        for line in f:
            row: Dict[str, Any] = json.loads(line)
            sys_answers[sys_name][row['doc_id']] = row['hypothesis']

In [7]:
# Display loading results (EDA)
print(f"[INFO] Successfully loaded {len(sys_answers)} system hypotheses.")
sample_sys: str = list(sys_answers.keys())[0]
sample_doc: str = list(sys_answers[sample_sys].keys())[0]
print(f"\n[DEBUG] Sample output for model '{sample_sys}':")
print(sys_answers[sample_sys][sample_doc][:150], "...")

[INFO] Successfully loaded 59 system hypotheses.

[DEBUG] Sample output for model 'CUNI-DocTransformer':
Hledáte informace v Cambridge
Hledáte <i>místo k pobytu</i>. Hotel by měl být v cenovém rozmezí <i>levný</i> a měl by <i>zahrnovat parkování zdarma</i ...


In [8]:
# Initialize containers for processing
records: List[Dict[str, Any]] = []
doc_id_maps: Dict[str, Dict[str, int]] = {'en-ko': {}, 'en-zh': {}} 
numeric_id_counters: Dict[str, int] = {'en-ko': 0, 'en-zh': 0}

In [9]:
with open(MAIN_FILE, 'r', encoding='utf-8') as f_main:
    for line in f_main:
        data: Dict[str, Any] = json.loads(line)
        original_doc_id: str = data['doc_id']
        
        # [FILTER 1] Include only 'general' collection (Drop others)
        if data.get('collection_id') != "general":
            continue
            
        # [FILTER 2] Include only specific domains (Exclude 'speech', etc.)
        domain: str = data.get('domain', 'unknown')
        if domain not in ["literary", "social", "news"]:
            continue
        
        lang_pair: Optional[str] = normalize_lang_pair(original_doc_id)
        if not lang_pair: continue 
            
        domain: str = data.get('domain', 'unknown')
        src_text: str = str(data.get('src_text', ''))
        ref_text: str = parse_reference_text(data.get('refs', {}))

        # Map doc_id to integer and save full context documents
        if original_doc_id not in doc_id_maps[lang_pair]:
            new_num_id: int = numeric_id_counters[lang_pair]
            doc_id_maps[lang_pair][original_doc_id] = new_num_id
            numeric_id_counters[lang_pair] += 1
            
            src_dir: str = os.path.join(BASE_OUT_DIR, lang_pair, 'src_docs')
            tgt_dir: str = os.path.join(BASE_OUT_DIR, lang_pair, 'tgt_docs')
            os.makedirs(src_dir, exist_ok=True)
            os.makedirs(tgt_dir, exist_ok=True)
            
            with open(os.path.join(src_dir, f"{new_num_id}.txt"), 'w', encoding='utf-8') as sf:
                sf.write(src_text)
            with open(os.path.join(tgt_dir, f"{new_num_id}.txt"), 'w', encoding='utf-8') as tf:
                tf.write(ref_text)
                
        new_doc_id: int = doc_id_maps[lang_pair][original_doc_id]
        
        # Segment text and join with system hypotheses
        src_segs: List[str] = src_text.split('\n')
        ref_segs: List[str] = ref_text.split('\n')
        
        for sys_name, answers_dict in sys_answers.items():
            hypothesis_text: str = str(answers_dict.get(original_doc_id, ""))
            if not hypothesis_text.strip(): continue
                
            hyp_segs: List[str] = hypothesis_text.split('\n')
            
            for idx in range(len(src_segs)):
                src_seg: str = src_segs[idx] if idx < len(src_segs) else ""
                tgt_seg: str = hyp_segs[idx] if idx < len(hyp_segs) else ""
                ref_seg: str = ref_segs[idx] if idx < len(ref_segs) else ""
                
                if not src_seg.strip(): continue
                
                records.append({
                    "lang_pair": lang_pair, "doc_id": new_doc_id, "domain": domain,
                    "system": sys_name, "src_seg": src_seg, "tgt_seg": tgt_seg, "ref_seg": ref_seg
                })

In [10]:
# Export mapping dictionary to JSON
for lp in doc_id_maps.keys():
    if doc_id_maps[lp]:
        with open(os.path.join(BASE_OUT_DIR, lp, 'doc_id.json'), 'w', encoding='utf-8') as f:
            json.dump(doc_id_maps[lp], f, ensure_ascii=False, indent=2)

print("[INFO] Data join and context document extraction completed.")

[INFO] Data join and context document extraction completed.


In [11]:
# Convert records to Pandas DataFrame for EDA
df_wmt = pd.DataFrame(records)

print(f"[INFO] Total merged segments: {len(df_wmt)}")
print("\n[DEBUG] Preview of the joined DataFrame:")
display(df_wmt.head())

[INFO] Total merged segments: 53724

[DEBUG] Preview of the joined DataFrame:


,lang_pair,doc_id,domain,system,src_seg,tgt_seg,ref_seg
0,en-ko,0,literary,Wenyiil,Rink Rats,링크 래츠,링크 랫츠
1,en-ko,0,literary,Wenyiil,Chapter 1: First Day,제 1장: 첫째 날,제1장: 첫날
2,en-ko,0,literary,Wenyiil,Kyle looked at his reflection in the mirror. H...,카일은 거울에 비친 자신의 모습을 바라보았다. 그는 얼굴에 새겨진 주름 하나하나를 ...,"카일은 거울에 비친 자신의 모습을 바라보았다. 얼굴에 새겨진, 지나간 세월을 떠올리..."
3,en-ko,0,literary,Wenyiil,He didn’t want to label it an identity crisis....,그는 그것을 정체성 위기라고 규정하고 싶지 않았다. 적어도 아직은 아니었다. 하지만...,여기에다 정체성의 위기라는 꼬리표를 붙이고 싶지는 않았다. 적어도 지금은 말이다. ...
4,en-ko,0,literary,Wenyiil,It had been a remarkable twenty-year pro caree...,그것은 대부분의 선수들이 꿈꿀 수 있는 놀라운 20년의 프로 경력이었다. 그는 자신...,22년간 프로 선수로서 남긴 그의 놀라운 족적은 대부분의 선수가 동경할 만한 것이었...


In [12]:
# Export to WMT24++ lab standard JSONL format
for lang_pair, group_df in df_wmt.groupby('lang_pair'):
    src_lang, tgt_lang = str(lang_pair).split('-')
    out_dir: str = os.path.join(BASE_OUT_DIR, str(lang_pair))
    
    in_path: str = os.path.join(out_dir, f'input_{lang_pair}.jsonl')
    out_path: str = os.path.join(out_dir, f'output_{lang_pair}.jsonl')
    
    sample_id: int = 1
    
    with open(in_path, 'w', encoding='utf-8') as f_in, \
         open(out_path, 'w', encoding='utf-8') as f_out:
        
        for _, row in group_df.iterrows():
            input_dict: Dict[str, Any] = {
                "sample_id": sample_id, "doc_id": row['doc_id'], "domain": row['domain'],
                "system": row['system'], "src_lang": src_lang, "tgt_lang": tgt_lang,
                "src_seg": row['src_seg'], "tgt_seg": row['tgt_seg']
            }
            output_dict: Dict[str, Any] = {
                "sample_id": sample_id, "src_seg": row['src_seg'], "tgt_seg": row['tgt_seg'],
                "ref_seg": row['ref_seg'], "human_pe_seg": row['ref_seg'], 
                "model_pe_seg": None, "manual": None, "auto": {}
            }
            
            f_in.write(json.dumps(input_dict, ensure_ascii=False) + '\n')
            f_out.write(json.dumps(output_dict, ensure_ascii=False) + '\n')
            sample_id += 1
            
    print(f"[SUCCESS] Exported {sample_id - 1} records for {lang_pair}.")

[SUCCESS] Exported 26136 records for en-ko.
[SUCCESS] Exported 27588 records for en-zh.


In [17]:
# [CORE] Extract base segments by dropping system-exploded duplicates
# This prevents counting the same segment multiple times (once for each system)
base_df = df_wmt.drop_duplicates(subset=['lang_pair', 'doc_id', 'src_seg']).copy()

# Calculate token length (using simple whitespace-based word count for approximation)
base_df['src_token_len'] = base_df['src_seg'].astype(str).apply(lambda x: len(x.split()))

In [19]:
# 1. The number of documents
num_docs = base_df['doc_id'].nunique()
print(f"1. The number of documents: {num_docs} documents")

1. The number of documents: 25 documents


In [20]:
# 2. The average number of segments per doc
avg_segs_per_doc = len(base_df) / num_docs if num_docs > 0 else 0
print(f"2. The average number of segments per doc: {avg_segs_per_doc:.2f} segments\n")

2. The average number of segments per doc: 58.08 segments



In [21]:
# 3. the average number of tokens per segment per domain
print("3. The average number of tokens per segment per domain:")
display(base_df.groupby('domain')['src_token_len'].mean().round(2).reset_index(name='avg_tokens_per_seg'))

3. The average number of tokens per segment per domain:


,domain,avg_tokens_per_seg
0,literary,26.96
1,news,93.86
2,social,33.94


In [22]:
# 4. The total number of segments per domain
print("\n4. The total number of segments per domain:")
display(base_df.groupby('domain').size().reset_index(name='total_segments'))


4. The total number of segments per domain:


,domain,total_segments
0,literary,736
1,news,190
2,social,526


In [23]:
# 5. The number of segments per system (will be the same)
sys_counts = df_wmt.groupby('system').size()
print("\n5. The number of segments per system:")
print(f"   -> {sys_counts.iloc[0]} segments (All {len(sys_counts)} systems have the exact same number)\n")


5. The number of segments per system:
   -> 1452 segments (All 38 systems have the exact same number)



In [24]:
# 6. The number of segments per language pair
print("6. The number of segments per language pair:")
display(base_df.groupby('lang_pair').size().reset_index(name='total_segments'))

6. The number of segments per language pair:


,lang_pair,total_segments
0,en-ko,726
1,en-zh,726


In [25]:
# 7. average token length per domain (Word character length)
base_df['src_char_len'] = base_df['src_seg'].astype(str).apply(lambda x: len(x.replace(" ", "")))
# Safe division to prevent ZeroDivisionError on empty segments
base_df['avg_char_per_token'] = base_df.apply(
    lambda row: row['src_char_len'] / row['src_token_len'] if row['src_token_len'] > 0 else 0, axis=1
)

print("\n7. Average token length per domain (Character length per token):")
display(base_df.groupby('domain')['avg_char_per_token'].mean().round(2).reset_index(name='avg_chars_per_token'))


7. Average token length per domain (Character length per token):


,domain,avg_chars_per_token
0,literary,4.73
1,news,5.01
2,social,4.95


In [26]:
# 8. The number of references per language pair
print("\n8. The number of references per language pair:")
print("   -> 1 reference (refA) per segment for both en-ko and en-zh\n")


8. The number of references per language pair:
   -> 1 reference (refA) per segment for both en-ko and en-zh



In [27]:
# 9. The number of systems
num_systems = df_wmt['system'].nunique()
print(f"9. The number of systems: {num_systems} systems")

9. The number of systems: 38 systems


In [28]:
base_df['src_token_len'] = base_df['src_seg'].astype(str).apply(lambda x: len(x.split()))
base_df['src_char_len'] = base_df['src_seg'].astype(str).apply(lambda x: len(x.replace(" ", "")))
base_df['avg_char_per_token'] = base_df.apply(
    lambda row: row['src_char_len'] / row['src_token_len'] if row['src_token_len'] > 0 else 0, axis=1
)

print("==================================================")
print("📊 [WMT 24/25 Data Statistics Final Report]")
print("==================================================\n")

# 1
num_docs = base_df['doc_id'].nunique()
print(f"1. The number of documents: {num_docs} documents\n")

# 2
print(f"2. The average number of segments per doc: {len(base_df) / num_docs if num_docs > 0 else 0:.2f} segments\n")

# 3
print("3. The average number of tokens per segment per domain:")
display(base_df.groupby('domain')['src_token_len'].mean().round(2).reset_index(name='avg_tokens_per_seg'))
print()

# 4
print("4. The total number of segments per domain:")
display(base_df.groupby('domain').size().reset_index(name='total_segments'))
print()

# 5
sys_counts = df_wmt.groupby('system').size()
print("5. The number of segments per system:")
print(f"   -> {sys_counts.iloc[0]} segments (All {len(sys_counts)} systems are strictly identical)\n")

# 6
print("6. The number of segments per language pair:")
display(base_df.groupby('lang_pair').size().reset_index(name='total_segments'))
print()

# 7
print("7. Average token length per domain (Character length per token):")
display(base_df.groupby('domain')['avg_char_per_token'].mean().round(2).reset_index(name='avg_chars_per_token'))
print()

# 8
print("8. The number of references per language pair:")
print("   -> 1 reference (refA) per segment for both en-ko and en-zh\n")

# 9
print(f"9. The number of systems: {df_wmt['system'].nunique()} systems\n")

print("==================================================")
print("✅ Report generation complete. Ready for GitHub PR.")
print("==================================================")

📊 [WMT 24/25 Data Statistics Final Report]

1. The number of documents: 25 documents

2. The average number of segments per doc: 58.08 segments

3. The average number of tokens per segment per domain:


,domain,avg_tokens_per_seg
0,literary,26.96
1,news,93.86
2,social,33.94



4. The total number of segments per domain:


,domain,total_segments
0,literary,736
1,news,190
2,social,526



5. The number of segments per system:
   -> 1452 segments (All 38 systems are strictly identical)

6. The number of segments per language pair:


,lang_pair,total_segments
0,en-ko,726
1,en-zh,726



7. Average token length per domain (Character length per token):


,domain,avg_chars_per_token
0,literary,4.73
1,news,5.01
2,social,4.95



8. The number of references per language pair:
   -> 1 reference (refA) per segment for both en-ko and en-zh

9. The number of systems: 38 systems

✅ Report generation complete. Ready for GitHub PR.
